#### P4-13a
# Model interpretation
#### M03

The final model is the gradient-boosting pipeline chosen in `13_model_comparison.ipynb`: `models/xgb/xgb_hotel_cancellation_pipeline.joblib`, validation ROC-AUC 0.7758.

## 1. Permutation importance

Importance is the drop in validation ROC-AUC when one column is shuffled. Five repeats, `random_state=42`. The test split is not used. These settings were already chosen on this validation split, so the ranking is slightly optimistic.

| Rank | Feature | Mean ROC-AUC drop | Std |
|---|---|---:|---:|
| 1 | `agent` | 0.0597 | 0.0016 |
| 2 | `lead_time` | 0.0550 | 0.0031 |
| 3 | `deposit_type` | 0.0233 | 0.0009 |
| 4 | `market_segment` | 0.0217 | 0.0018 |
| 5 | `previous_bookings_not_canceled` | 0.0192 | 0.0017 |
| 6 | `lead_per_night` | 0.0092 | 0.0011 |
| 7 | `adr_ratio` | 0.0035 | 0.0007 |
| 8 | `total_nights` | 0.0029 | 0.0008 |

Every other column drops the score by less than 0.003.

## 2. What the top features mean

On the validation split, about 29.6% of bookings cancel.

- `agent` is the travel agency that made the booking. Agency 9 cancels at 44% (3,948 bookings). Bookings with no agency cancel at 13% (1,472 bookings). The model does not see the raw ID. It sees that agency's past cancellation rate.
- `lead_time` is how many days ahead the guest booked. The shortest quarter cancels at about 12%. The longest quarter cancels at about 42%.
- `deposit_type` is the deposit rule already on the booking. All 127 non-refundable validation bookings cancelled. No-deposit bookings cancel at 29%.
- `market_segment` is how the booking was sold. Online travel agents cancel at 38%. That is the large high-rate group.
- `previous_bookings_not_canceled` counts earlier stays that were kept. Guests with at least one kept stay cancel at 4% (491 bookings). Guests with none cancel at 31%.
- `lead_per_night` is lead time divided by the length of the stay. It repeats the lead-time pattern: a long gap before a short stay cancels more.
- `adr_ratio` is the room price divided by the usual training-set price for that hotel, room type, and month. Bookings priced above that usual level cancel a little more (about 30% against 21%).
- `total_nights` is the length of the stay. Longer stays cancel a little more (about 32% against 24%).

## 3. Proxies

- `agent` and `company` are ID codes that were target-encoded. A high `agent` importance stands in for the sales channel and segment, not for a guest habit the hotel can change. `company` itself barely moves the score.
- `deposit_type` is a policy the hotel already set. Non-refundable is not a new reason to ask for a deposit. It is the rule that was already applied, and in this split those bookings all cancelled.
- `market_segment` and `distribution_channel` describe the same booking path. Once segment is known, channel adds only a 0.0024 drop.
- `lead_time` and `lead_per_night` share one timing story. `total_nights` shares its story with the weekend-night and weekday-night columns.
- `adr`, `adr_per_guest`, and `adr_ratio` are three views of price. `adr_ratio` also carries hotel, room type, and arrival month, because the reference price is the training median of that group.
- `arrival_date_year` is not in the model. It only marks when the data were collected.

